# 04 - 完整 BCNLT 流水线

本 Notebook 演示端到端的 BCNLT 预处理流水线：
1. 使用 `BCNLTPreprocessor` 类（统一 sklearn 风格 API）
2. fit 训练集 → transform 测试集
3. 与 PCA 进行初步对比
4. 验证流水线正确性

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

from data.load_dataset import load_orl, train_test_split_orl
from src.pipeline import BCNLTPreprocessor
from src.utils import evaluate_classifier, apply_pca, show_reconstruction_comparison

X, y = load_orl(data_dir='../data/ORL')
X_train, X_test, y_train, y_test = train_test_split_orl(X, y, n_train=5)
IMG_SHAPE = (64, 64)
print(f'训练集: {X_train.shape}, 测试集: {X_test.shape}')

训练集: (200, 4096), 测试集: (200, 4096)


## 1. 运行 BCNLT 完整流水线

In [2]:
import time

preprocessor = BCNLTPreprocessor(
    n_row_clusters=4,
    n_col_clusters=4,
    cluster_method='spectral',
    poly_degree=2,
    use_nonlinear=False,
    verbose=True,
    random_state=42
)

t0 = time.time()
X_train_proc = preprocessor.fit_transform(X_train)
t_fit = time.time() - t0

t0 = time.time()
X_test_proc = preprocessor.transform(X_test)
t_transform = time.time() - t0

print(f'\n训练时间: {t_fit:.2f}s')
print(f'测试变换时间: {t_transform:.4f}s')
print(f'处理后训练集: {X_train_proc.shape}')
print(f'处理后测试集: {X_test_proc.shape}')

[BCNLT] Step 1/3: 双向聚类...


双向聚类结果 (method=spectral)
  行聚类数 k_r = 4
    簇 0: 19 个样本
    簇 1: 51 个样本
    簇 2: 61 个样本
    簇 3: 69 个样本
  列聚类数 k_c = 4
    簇 0: 1069 个特征
    簇 1: 1160 个特征
    簇 2: 246 个特征
    簇 3: 1621 个特征
[BCNLT] Step 2/3: 对角矩阵优化与块重构...
[BCNLT]   重构相对误差: 0.1558  PSNR: 20.99 dB
[BCNLT] Step 3/3: 非线性变换学习...


[BCNLT] 拟合完成。

训练时间: 4.71s
测试变换时间: 0.0298s
处理后训练集: (200, 4096)
处理后测试集: (200, 4096)


## 2. 分类评估

In [3]:
# BCNLT 预处理后的分类精度
acc_bcnlt_svm = evaluate_classifier(X_train_proc, y_train, X_test_proc, y_test, 'svm')
acc_bcnlt_knn = evaluate_classifier(X_train_proc, y_train, X_test_proc, y_test, 'knn')

# 无预处理（原始特征）
acc_raw_svm = evaluate_classifier(X_train, y_train, X_test, y_test, 'svm')
acc_raw_knn = evaluate_classifier(X_train, y_train, X_test, y_test, 'knn')

# PCA 降维
X_train_pca, X_test_pca = apply_pca(X_train, X_test, n_components=80)
acc_pca_svm = evaluate_classifier(X_train_pca, y_train, X_test_pca, y_test, 'svm')
acc_pca_knn = evaluate_classifier(X_train_pca, y_train, X_test_pca, y_test, 'knn')

print('方法           SVM        KNN')
print('-' * 40)
print(f'Raw         {acc_raw_svm*100:.2f}%    {acc_raw_knn*100:.2f}%')
print(f'PCA(80)     {acc_pca_svm*100:.2f}%    {acc_pca_knn*100:.2f}%')
print(f'BCNLT       {acc_bcnlt_svm*100:.2f}%    {acc_bcnlt_knn*100:.2f}%')

方法           SVM        KNN
----------------------------------------
Raw         92.00%    89.00%
PCA(80)     93.00%    87.50%
BCNLT       89.50%    88.50%


In [4]:
# 可视化对比
methods = ['Raw', 'PCA(80)', 'BCNLT']
svm_accs = [acc_raw_svm*100, acc_pca_svm*100, acc_bcnlt_svm*100]
knn_accs = [acc_raw_knn*100, acc_pca_knn*100, acc_bcnlt_knn*100]

x = np.arange(len(methods))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 5))
bars1 = ax.bar(x - width/2, svm_accs, width, label='SVM', color='#4c72b0', alpha=0.85)
bars2 = ax.bar(x + width/2, knn_accs, width, label='KNN', color='#dd8452', alpha=0.85)

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(methods, fontsize=11)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('ORL Recognition Accuracy (5 train / 5 test per class)', fontsize=11)
ax.set_ylim(0, 110)
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/04_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\86175\AppData\Local\Temp\ipykernel_14632\2594450974.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. 测试集变换效果可视化

In [5]:
# 展示测试集图像预处理前后对比
fig = show_reconstruction_comparison(
    X_test, X_test_proc, y_test,
    img_shape=IMG_SHAPE,
    n_samples=8,
    title='Test Set: Original vs BCNLT Preprocessed',
    save_path='../results/figures/04_test_transform.png'
)
plt.show()

C:\Users\86175\AppData\Local\Temp\ipykernel_14632\258435376.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# 检查行簇分配（测试样本被分到哪个样本簇）
test_row_labels = preprocessor._clusterer.assign_row_cluster(X_test)
train_row_labels = preprocessor.row_labels_

print('测试集样本簇分配：')
for a in range(preprocessor.n_row_clusters):
    cnt_train = (train_row_labels == a).sum()
    cnt_test  = (test_row_labels == a).sum()
    print(f'  簇 {a}: 训练={cnt_train} 样本, 测试={cnt_test} 样本')

测试集样本簇分配：
  簇 0: 训练=19 样本, 测试=19 样本
  簇 1: 训练=51 样本, 测试=48 样本
  簇 2: 训练=61 样本, 测试=64 样本
  簇 3: 训练=69 样本, 测试=69 样本
